In [ ]:
from __future__ import annotations

import pickle
import fsspec

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA

import manifold_dynamics.neural_utils as nu
import manifold_dynamics.paths as pth
import manifold_dynamics.spike_response_stats as srs
import manifold_dynamics.tuning_utils as tut
import visionlab_utils.storage as vst

In [ ]:
# ROI name, should be 3 parts (index.label.category)
target = "19.Unknown.F"
target_parts = target.split(".")
roi_label = f"{int(target_parts[0]):02d}.{target_parts[1]}.{target_parts[2]}"

# load in multi-session ROI data, binned to PSTH
raster_4d = nu.significant_trial_raster(target, alpha=0.05, bin_size_ms=20)

# get top-k value for ROI
topk_local = vst.fetch(f"{pth.OTHERS}/topk_vals.pkl")
with open(topk_local, "rb") as f:
    topk_vals = pickle.load(f)
top_k = int(topk_vals[roi_label]["k"])

print(f"Resolved ROI target: {target}")
print(f"Using top-k = {top_k}")
print(f"Raster shape after binning {raster_4d.shape}")

In [ ]:
raster_3d = np.nanmean(raster_4d, axis=3)
scores = tut.rank_images_by_response(raster_3d)

idx_topk = scores[:top_k]
trial_3d = raster_3d[:, :, idx_topk]
print(f"Trial averaged, top-k shape {trial_3d.shape}")

In [ ]:
# load in alexnet embeddings
feature_uri = f"{pth.SAVEDIR}/alexnet/alexnet_acts.pkl"
acts_local = vst.fetch(feature_uri)
with open(acts_local, "rb") as f:
    acts = pickle.load(f)

layers = ['classifier.2', 'classifier.5']
for layer_key in layers:
    arr = acts[layer_key]
    if hasattr(arr, "detach"):
        arr = arr.detach().cpu().numpy()
        feature_matrix = np.asarray(arr)


    if feature_matrix.ndim != 2:
        raise ValueError(f'expected 2d feature matrix, got shape {feature_matrix.shape}')
    print(f"{layer_key} activation shape {feature_matrix.shape}")
    
    # basic sanity check
    if feature_matrix.ndim != 2:
        raise ValueError(f'expected 2d feature matrix, got shape {feature_matrix.shape}')
    
    # fit pca on all images
    pca = PCA(n_components=2)
    Z = pca.fit_transform(feature_matrix)  # (images, 2)
    
    # select roi-preferred images
    Z_topk = Z[idx_topk]
    centroid = np.nanmean(Z_topk, axis=0)
    
        # --- figure layout: left = PCA, right = 3x3 images ---
    fig = plt.figure(figsize=(10, 5))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.5, 1.0])
    
    # ---- PCA plot ----
    ax_pca = fig.add_subplot(gs[0, 0])
    
    ax_pca.scatter(Z[:, 0], Z[:, 1], s=10, alpha=0.1, c='black', label='all images',)
    ax_pca.scatter(Z_topk[:, 0], Z_topk[:, 1], s=30, alpha=0.75, label='top-k')
    ax_pca.scatter(centroid[0], centroid[1], s=120, marker='X', label='centroid')
    
    ax_pca.set_xlabel(f'PC1')
    ax_pca.set_ylabel(f'PC2')
    ax_pca.set_title(f'{target} in {layer_key} space')
    ax_pca.legend(frameon=False)
    
    sns.despine(ax=ax_pca, trim=True)
    
    # ---- image grid ----
    gs_imgs = gs[0, 1].subgridspec(3, 3)
    
    # take first 9 top-k images (or fewer if needed)
    n_show = min(9, len(idx_topk))
    
    for i in range(9):
        ax = fig.add_subplot(gs_imgs[i // 3, i % 3])
        ax.axis('off')
    
        if i < n_show:
            nu.plot_stimulus_image(idx_topk[i], ax=ax)
        # ax.set_title(f'{idx_topk[i]}')
    
    plt.tight_layout()
    plt.show()